## Settings and imports

In [1]:
import numpy as np
import pandas as pd
import os
import torch
import matplotlib.pyplot as plt
import string

## Loading the data

In [2]:
data_path = '/data/DANE.xlsx'

In [3]:
df = pd.ExcelFile(os.getcwd() + data_path)
inks_df = df.parse('a.', header=0, index_col=0, usecols=list) #inKs
inds_df = df.parse('i.', header=0, index_col=0, usecols=list)  #inDs

In [4]:
#[1425:] rzeczywiste

## Preprocessing

### Some checks on the data

1. Let's check if column naming is consistent (inDs).

In [5]:
series_based_on_sample_name = pd.Series(inds_df.index).apply(
                                lambda x: x[:-2] if x[-2].isdigit() else x[:-1]).reset_index(drop=True)

In [6]:
series_based_on_column = inds_df['seria'].reset_index(drop=True)

In [7]:
np.all(series_based_on_column == series_based_on_sample_name)

True

Everything is fine. For inKs:

In [8]:
series_based_on_sample_name = pd.Series(inks_df.index).apply(
                                lambda x: x[:-2] if x[-2].isdigit() else x[:-1]).reset_index(drop=True)

In [9]:
series_based_on_column = inks_df['seria'].reset_index(drop=True)

In [10]:
np.all(series_based_on_column == series_based_on_sample_name)

True

Everything ok.

2. Let's check if naming is consistent across two sheets (i.e. for inDs and for inKs).

In [11]:
series_from_inds_sheet = inds_df['seria'].apply(lambda x: x[:-1]).reset_index(drop=True)

In [12]:
series_from_inks_sheet = inks_df['seria'].apply(lambda x: x[:-1]).reset_index(drop=True)

In [13]:
np.all(series_from_inds_sheet == series_from_inks_sheet)

False

It's not consistent, but it's not the problem, cause rows in both tables correspond to each other.

In [14]:
inks_df.shape == inds_df.shape

True

### Joining corresponding samples from inds and inks into one table

In [15]:
inks_df = inks_df.reset_index(drop=False)

In [16]:
inds_df = inds_df.reset_index(drop=False)

In [17]:
inDKs_df = inds_df.join(inks_df, how='inner', rsuffix='_inds', lsuffix='_inks')

In [18]:
# with pd.option_context("display.max_rows", inDKs_df.shape[0]):
#     display(inDKs_df[['nazwa próbki_inds', 'nazwa próbki_inks']][
#                 (inDKs_df['nazwa próbki_inds'].apply(
#                         lambda x: x[:4]) != inDKs_df['nazwa próbki_inks'].apply(
#                                         lambda x: x[:4]))
#                                                         ])

### Removing some data

1. Let's remove rows with missing values.

In [19]:
(inDKs_df.shape[0] - inDKs_df.dropna().shape[0])/inDKs_df.shape[0]

0.030303030303030304

In [20]:
inDKs_df.dropna(inplace=True)

In [21]:
inDKs_df.shape

(3840, 64)

2. Let's keep only columns that we need.

In [22]:
inks_colnames = [col for col in inDKs_df.columns if (col.endswith('_inks') and 
                                                     col != 'nazwa próbki_inks'and
                                                    col != 'seria_inks' and
                                                    col != 'Fe_inks')]
inds_colnames = [col for col in inDKs_df.columns if (col.endswith('_inds') and 
                                                     col != 'nazwa próbki_inds'and
                                                    col != 'seria_inds' and
                                                    col != 'Fe_inds')]
#no iron because it's a reference element

### Creating features and labels matrices

In [23]:
X = inDKs_df[inds_colnames]
y = inDKs_df[inks_colnames]

## Building neural network

## Training

## Testing

## Interpretability